In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# CARICAMENTO DATI (Dalla cartella 'data median')
path_train = 'android_session_1_1_median.csv'
path_test  = 'ios_session_1_3_median.csv'

print(f"Caricamento Training: {path_train}")
df_train = pd.read_csv(path_train).fillna(-110)

print(f"Caricamento Test: {path_test}")
df_test = pd.read_csv(path_test).fillna(-110)

# Allineamento colonne
# Android e iOS potrebbero aver visto beacon diversi.
# Bisogna garantire che il modello veda le stesse colonne nello stesso ordine.

# Prendiamo solo le colonne beacon (quelle col trattino '-')
beacon_cols = [c for c in df_train.columns if '-' in c and c not in ['room', 'artwork']]
target_col = 'room' # Testiamo sulla STANZA che è più robusto dell'opera

print(f"Numero beacon usati per il training: {len(beacon_cols)}")

X_train = df_train[beacon_cols]
y_train = df_train[target_col]

# Forziamo il Test set ad avere le stesse colonne del Train
# Se manca una colonna, viene riempita con -110. Se ce ne sono in più, vengono ignorate.
X_test = df_test.reindex(columns=beacon_cols, fill_value=-110)
y_test = df_test[target_col]

# Training & evaluation
print("\nAddestramento Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("Predizione su iOS...")
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("\n" + "="*40)
print(f"ACCURATEZZA CROSS-PLATFORM: {acc*100:.2f}%")
print("="*40)

print("\nReport per Classe:")
print(classification_report(y_test, y_pred))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=rf.classes_, yticklabels=rf.classes_)
plt.title(f'Confusion Matrix Cross-Platform (Acc: {acc*100:.1f}%)')
plt.ylabel('Vera Stanza (iOS)')
plt.xlabel('Stanza Predetta (Modello Android)')
plt.savefig('confusion_matrix_cross_platform.png')
print("Grafico salvato: confusion_matrix_cross_platform.png")